In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from sklearn.model_selection import train_test_split

import config
from src.db_io import leer_tabla_sqlite
from src.features_modelo import features_modelo_a
from src.fuga import validar_sin_fuga

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")

# SPEC_V2 §2: entrenamiento sobre toda la base apta. Los clientes sin productos
# son ejemplos negativos legítimos y necesarios, no se excluyen.
entrenables = df[df["apto_entrenamiento"] == 1].reset_index(drop=True)

feature_cols = features_modelo_a(entrenables.columns)   # lanza si hay fuga
X = pd.get_dummies(
    entrenables[feature_cols],
    columns=[c for c in ["desc_segmento", "grupo_edad", "desc_tipo_de_vivienda"]
             if c in feature_cols],
    dummy_na=False,
)
y = entrenables["etiqueta_adopcion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
print(f"entrenables: {len(entrenables)} de {len(df)}")
print(f"train: {X_train.shape}, test: {X_test.shape}, tasa adopción train: {y_train.mean():.4f}")


entrenables: 860153 de 860223
train: (688122, 73), test: (172031, 73), tasa adopción train: 0.0717


In [2]:
# SPEC_V2 §1.3: el guard es el mismo que cubre tests/test_fuga.py.
# Se ejecuta sobre las columnas REALES que entran al fit (post get_dummies),
# no solo sobre la lista previa.
validar_sin_fuga(X_train.columns, contexto="fit del modelo de propensión")
print(f"OK: {X_train.shape[1]} columnas de entrenamiento, ninguna derivada de la etiqueta")


OK: 73 columnas de entrenamiento, ninguna derivada de la etiqueta


In [3]:
import json
import joblib
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score

# HistGradientBoostingClassifier maneja NaN nativamente (SPEC_V2 §3.2 depende de
# esto): no hace falta imputar los ~190 clientes con financieros nulos.
modelo = HistGradientBoostingClassifier(random_state=config.RANDOM_STATE)
modelo.fit(X_train, y_train)

proba = modelo.predict_proba(X_test)[:, 1]
pred = modelo.predict(X_test)

metricas = {
    "auc": float(roc_auc_score(y_test, proba)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "n_train": int(len(X_train)),
    "n_test": int(len(X_test)),
    "n_features": int(X_train.shape[1]),
}
print(metricas)

# SPEC_V2 §1: por encima de 0.95 se asume fuga residual y se detiene el trabajo.
assert metricas["auc"] <= config.UMBRAL_AUC_FUGA, (
    f"AUC={metricas['auc']:.4f} > {config.UMBRAL_AUC_FUGA}: sospecha de fuga residual. "
    "Investigar antes de continuar (SPEC_V2 §1)."
)

(config.OUTPUTS_DIR / "models").mkdir(parents=True, exist_ok=True)
joblib.dump(modelo, config.OUTPUTS_DIR / "models" / "propension_adopcion.pkl")
with open(config.OUTPUTS_DIR / "models" / "metricas_propension.json", "w") as f:
    json.dump(metricas, f, indent=2)


{'auc': 0.8942246185377238, 'precision': 0.58, 'recall': 0.07528190151699521, 'n_train': 688122, 'n_test': 172031, 'n_features': 73}


In [4]:
import numpy as np

# El AUC mide calidad de ranking; el recall al umbral por defecto (0.5) es bajo
# porque la tasa base de adopción es ~7-8% y el modelo rara vez supera 0.5 de
# probabilidad. Para un caso de uso de targeting de campaña, lo relevante no es
# el umbral fijo sino: "si contacto el top N% de clientes mejor puntuados, ¿qué
# recall/precisión obtengo?" — la métrica natural para priorizar contactos.
orden = np.argsort(-proba)
proba_ordenada = proba[orden]
y_ordenado = y_test.to_numpy()[orden]
n = len(y_test)

filas = []
for pct in [0.01, 0.05, 0.10, 0.20]:
    corte = max(1, int(np.ceil(n * pct)))
    umbral = proba_ordenada[corte - 1]
    seleccionados = y_ordenado[:corte]
    precision_topn = seleccionados.sum() / corte
    recall_topn = seleccionados.sum() / y_test.sum()
    filas.append({
        "top_pct": pct,
        "n_contactados": corte,
        "umbral_probabilidad": umbral,
        "precision": precision_topn,
        "recall": recall_topn,
    })

curva_top_n = pd.DataFrame(filas)
print(curva_top_n.to_string(index=False))

curva_top_n.to_csv(config.OUTPUTS_DIR / "models" / "curva_precision_recall.csv", index=False)

# Interpolado desde `metricas` (celda anterior), nunca hardcodeado: un número fijo
# aquí queda desactualizado en cuanto cambie algo aguas arriba (fue exactamente
# el defecto detectado en la revisión de esta gate).
print(
    f"\nInterpretación: el AUC alto ({metricas['auc']:.2f}) indica que el modelo ordena bien a los "
    f"clientes por probabilidad de adopción, aun cuando el recall al umbral 0.5 sea "
    f"bajo ({metricas['recall']:.0%}). Para targeting de campaña -contactar el top N% de clientes con mayor "
    "score en vez de aplicar un umbral fijo de probabilidad- el modelo es útil: por "
    "ejemplo, contactando el 10% con mayor score se captura una fracción muy superior "
    "de adoptantes reales que un contacto aleatorio del mismo tamaño."
)

 top_pct  n_contactados  umbral_probabilidad  precision   recall
    0.01           1721             0.494210   0.578152 0.080717
    0.05           8602             0.342577   0.442688 0.308915
    0.10          17204             0.252091   0.370205 0.516671
    0.20          34407             0.133221   0.277473 0.774479

Interpretación: el AUC alto (0.89) indica que el modelo ordena bien a los clientes por probabilidad de adopción, aun cuando el recall al umbral 0.5 sea bajo (8%). Para targeting de campaña -contactar el top N% de clientes con mayor score en vez de aplicar un umbral fijo de probabilidad- el modelo es útil: por ejemplo, contactando el 10% con mayor score se captura una fracción muy superior de adoptantes reales que un contacto aleatorio del mismo tamaño.
